# Data Cleaning Pipeline

In [1]:
%load_ext autoreload
%autoreload 2

from io_utils import *
from clean import *

import matplotlib.pyplot as plt

## Load the Data

In [2]:
df = load_raw_data()
df.head()

,PatNum,EmplType,Certification,InjuryMech,High_impact_InjSev,Amnesia_verb,LOCSeparate,LocLen,Seiz,SeizOccur,...,Finding20,Finding21,Finding22,Finding23,DeathTBI,HospHead,HospHeadPosCT,Intub24Head,Neurosurgery,PosIntFinal
0,1,3.0,3,11.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,92,0.0,0.0,0,0.0,0.0,0.0
1,2,5.0,3,8.0,2.0,0.0,0.0,92.0,0.0,92.0,...,0,0,0,0,0.0,0.0,0,0.0,0.0,0.0
2,3,5.0,3,5.0,2.0,NaN,NaN,92.0,NaN,92.0,...,0,0,0,0,0.0,1.0,0,0.0,0.0,0.0
3,4,5.0,3,6.0,1.0,91.0,0.0,92.0,0.0,92.0,...,92,92,92,92,0.0,0.0,0,0.0,0.0,0.0
4,5,3.0,3,12.0,2.0,91.0,0.0,92.0,0.0,92.0,...,0,0,0,0,0.0,0.0,0,0.0,0.0,0.0


In [3]:
# print the basic information of the dataset
print("Dataset shape:", df.shape)
print("\nDataset columns:", df.columns)
print("\nData types of each column:")
print(df.dtypes)

# check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nPercentage of missing values per column:")
print((df.isnull().sum() / len(df)) * 100)

# check for duplicates
print("\nUnique values per column:")
print(df.nunique())

# check for summary statistics of numerical columns
print("\nSummary statistics:")
print(df.describe())


Dataset shape: (43399, 125)

Dataset columns: Index(['PatNum', 'EmplType', 'Certification', 'InjuryMech',
       'High_impact_InjSev', 'Amnesia_verb', 'LOCSeparate', 'LocLen', 'Seiz',
       'SeizOccur',
       ...
       'Finding20', 'Finding21', 'Finding22', 'Finding23', 'DeathTBI',
       'HospHead', 'HospHeadPosCT', 'Intub24Head', 'Neurosurgery',
       'PosIntFinal'],
      dtype='str', length=125)

Data types of each column:
PatNum                  int64
EmplType              float64
Certification           int64
InjuryMech            float64
High_impact_InjSev    float64
                       ...   
HospHead              float64
HospHeadPosCT           int64
Intub24Head           float64
Neurosurgery          float64
PosIntFinal           float64
Length: 125, dtype: object

Missing values per column:
PatNum                  0
EmplType               18
Certification           0
InjuryMech            301
High_impact_InjSev    334
                     ... 
HospHead                

## Data Cleaning: Phase 0 — Column Renaming

In [4]:
df_renamed = rename_columns(df)
summarize_rename(df, df_renamed)

{'n_rows_before': 43399,
 'n_cols_before': 125,
 'n_rows_after': 43399,
 'n_cols_after': 125,
 'unchanged_cols': []}

# Data Cleaning: Phase 1 — Structural Validation

### Convert Type

In [5]:
non_numeric_columns = identify_non_numeric_columns(df_renamed)
non_numeric_columns

[]

### Validate Allowed Values

In [6]:
issues = validate_allowed_values(df_renamed)
issues

,column,invalid_value,n,examples


### Check Duplicate Row

In [7]:
dup_summary = check_duplicates(df_renamed, key_col="patient_id")
dup_summary

{'n_dup_key_rows': 0, 'n_dup_key_values': 0, 'n_dup_rows': 0}

## Data Cleaning -- Inconsistency Check

In [8]:
issues = summarize_consistency(df_renamed)
save_table(issues, "consistency_issues.csv")
issues

,group,rule,n_inconsistent,pct_rows,examples
0,Parent-child Logic,loss_of_consciousness=1 but loc_duration missing,1536,3.539252,"[49, 54, 65, 92, 142]"
1,Parent-child Logic,ct_planned=1 but no CT reason is marked,1406,3.239706,"[67, 79, 94, 141, 144]"
2,Parent-child Logic,headache=1 but headache_onset missing,1332,3.069195,"[7, 11, 28, 166, 327]"
3,Parent-child Logic,headache=1 but headache_severity missing,1044,2.405585,"[28, 60, 82, 166, 194]"
4,Parent-child Logic,scalp_hematoma=1 but hematoma_size missing,742,1.709717,"[163, 330, 335, 351, 423]"
5,Parent-child Logic,ams=1 but no AMS subtype marked,615,1.417083,"[167, 267, 304, 412, 539]"
6,Parent-child Logic,trauma_above_clavicles=1 but no location marked,302,0.695869,"[11, 21, 196, 279, 280]"
7,Parent-child Logic,scalp_hematoma=1 but hematoma_location missing,211,0.486186,"[466, 596, 718, 1148, 1369]"
8,CT & Outcomes,severe outcome component but citbi!=1,160,0.368672,"[2, 304, 312, 466, 497]"
9,Parent-child Logic,seizure=1 but seizure_duration missing,117,0.269591,"[75, 477, 670, 754, 1481]"


## Data Cleaning -- Missing Value

### Missing Overview

In [9]:
missing_na = df_renamed.isna().mean().sort_values(ascending=False)
missing_na.head(20)

dizziness                    0.368027
ethnicity                    0.367889
acting_normal                0.076845
race                         0.073919
loc_duration                 0.058895
observed_in_ed               0.054748
amnesia_present              0.052904
loss_of_consciousness        0.043595
drug_or_alcohol_suspicion    0.041890
headache_onset               0.030692
gcs_motor                    0.030185
gcs_verbal                   0.029909
gcs_eye                      0.029678
headache_severity            0.024056
vomiting_last                0.022858
ct_sedation                  0.021060
posttraumatic_seizure        0.021014
scalp_hematoma_size          0.017097
neuro_deficit                0.015208
headache                     0.015023
dtype: float64

In [10]:
special_missing = summarize_special_missing(df_renamed)
special_missing.sort_values("proportion", ascending=False).head(20)

,column,code,proportion
20,palpable_skull_fracture_depressed,92,0.994839
21,basilar_hemotympanum,92,0.990875
22,basilar_otorrhea,92,0.990875
24,basilar_retroauricular_ecchymosis,92,0.990875
25,basilar_rhinorrhea,92,0.990875
23,basilar_periorbital_ecchymosis,92,0.990875
7,seizure_timing,92,0.986106
8,seizure_duration,92,0.986106
66,ct_sedation_other,92,0.984931
65,ct_sedation_tech_request,92,0.984931


In [11]:
total_missing = summarize_total_missing(df_renamed)
total_missing.head(20)

,column,total_missing_proportion
35,palpable_skull_fracture_depressed,0.996106
38,basilar_hemotympanum,0.990875
39,basilar_otorrhea,0.990875
40,basilar_periorbital_ecchymosis,0.990875
41,basilar_retroauricular_ecchymosis,0.990875
42,basilar_rhinorrhea,0.990875
10,seizure_duration,0.988802
9,seizure_timing,0.987742
89,ct_sedation_other,0.984931
88,ct_sedation_tech_request,0.984931


### Check Structural Missing

In [12]:
pd.crosstab(
    df_renamed["posttraumatic_seizure"],
    df_renamed["seizure_timing"]
)

seizure_timing,1.0,2.0,3.0,92.0
posttraumatic_seizure,,,,
0.0,0,0,0,41884
1.0,253,210,69,0


### Check Informative Missing 

In [13]:
pd.crosstab(
    df_renamed["age_two_plus"],
    df_renamed["dizziness"].isna()
)

dizziness,False,True
age_two_plus,,
1,1360,9544
2,26067,6428


In [14]:
pd.crosstab(df_renamed["dizziness"].isna(),
            df_renamed["citbi"],
            normalize="index")

citbi,0.0,1.0
dizziness,,
False,0.992924,0.007076
True,0.964355,0.035645


### Check MCAR

In [15]:
# ethnicity missing vs outcome
pd.crosstab(df_renamed["ethnicity"].isna(),
            df_renamed["citbi"],
            normalize="index")


citbi,0.0,1.0
ethnicity,,
False,0.98432,0.01568
True,0.97913,0.02087


In [16]:

pd.crosstab(df_renamed["ethnicity"].isna(),
            df_renamed["gcs_group"],
            normalize="index")

gcs_group,1,2
ethnicity,,
False,0.019903,0.980097
True,0.026494,0.973506


### Handle Missing Values

In [17]:
df_cleaned = handle_missing_values(df_renamed)

In [18]:
print("Before cleaning:")
print("Shape:", df_renamed.shape)
print("After cleaning:")
print("Shape:", df_cleaned.shape)

Before cleaning:
Shape: (43399, 125)
After cleaning:
Shape: (43399, 125)


## Data Cleaning -- Column Reduction

In [19]:
plan = build_drop_plan(
    df_cleaned,
    timepoint="pre_ct",
    drop_ethnicity=True,
    high_missing_threshold=0.95,
    keep_age="age_years",
    keep_gcs="gcs_total",
)

df_drop, dropped_cols = apply_drop_plan(df_cleaned, plan)
df_drop.shape

(43399, 75)

### Data Cleaning -- Comparison with Published Rates 

In [20]:
mine, paper = compare_with_kuppermann_2009(df_cleaned)
save_table(mine, "my_data_cleaning_summary.csv")
save_table(paper, "paper_data_summary.csv")
mine, paper

(                             my_ct_done_%  my_tbi_on_ct_among_ct_%  \
 Age <2 (my data, GCS 14-15)         31.05                     8.53   
 Age ≥2 (my data, GCS 14-15)         36.75                     4.28   
 
                              my_citbi_%  my_neurosurgery_%  
 Age <2 (my data, GCS 14-15)        0.91               0.18  
 Age ≥2 (my data, GCS 14-15)        0.88               0.13  ,
                      paper_tbi_on_ct_%  paper_citbi_%  paper_neurosurgery_%
 Age <2 (Derivation)                8.1            0.9                   0.2
 Age <2 (Validation)                9.8            1.1                   0.2
 Age ≥2 (Derivation)                4.1            0.9                   0.1
 Age ≥2 (Validation)                5.2            1.0                   0.2)

In [21]:
df = load_raw_data()
df_cleaned = clean_data(df)
df_cleaned_without_drops = clean_data_without_dropping(df)


save_table(df_cleaned, "df_cleaned.csv")
save_table(df_cleaned_without_drops, "df_cleaned_without_drops.csv")

PosixPath('/Users/ren/Library/CloudStorage/OneDrive-Personal/2025-26 UC Berkeley/2025-26 Term 2/STAT 214 Data Analysis and Machine Learning for Real-World Decision Making/stat-214/lab1/data/df_cleaned_without_drops.csv')